# 00. 環境チェック

このノートブックが最後まで緑になるまで、他のノートブックに進まないこと。
ここで落ちる項目は、そのまま Web アプリでも落ちる。

確認するのは 5 つ:

1. Ollama に到達できるか / 使うモデルが pull 済みか
2. リソースポリシー（商用限定）が読めるか
3. biomni がインストールされているか
4. **biomni の `default_config` が Ollama を向いているか**（docs/design/04 §4.3）
5. データレイクに必要なファイルがあるか

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from biomni_hypo.config import Settings, apply_biomni_env
from biomni_hypo.llm import ollama_status
from biomni_hypo.policy import ResourcePolicy

settings = Settings()
print("model            :", settings.model)
print("ollama_base_url  :", settings.ollama_base_url)
print("num_ctx          :", settings.num_ctx)
print("data_path        :", settings.data_path)
print("use_tool_retriever:", settings.use_tool_retriever)
print("commercial_mode  :", settings.commercial_mode)

## 1. Ollama

`reachable=False` なら `ollama serve` が動いていない。
使うモデルが一覧に無ければ `ollama pull qwen3:14b`。

In [ ]:
status = ollama_status(settings.ollama_base_url)
print("reachable:", status.reachable, status.error)
print("models   :", status.models)
print()
if status.reachable and settings.model not in status.models:
    print(f"⚠️  {settings.model} が未取得です:  ollama pull {settings.model}")

## 2. リソースポリシー（商用限定）

`A1(commercial_mode=True)` はデータセットしか絞らない。ツールは絞らないので、
このポリシーが商用利用の実質的な担保になる（docs/design/05）。

In [ ]:
policy = ResourcePolicy.load(settings.policy_path)
print("policy version :", policy.version, "/ mode:", policy.mode)
print("許可データセット:", len(policy.allowed_dataset_names()))
print("拒否ツール      :", policy.denied_tool_names())
print("許可モデル      :", policy.allowed_model_names())
print()

d = policy.check_model(settings.model)
print(f"モデル {settings.model}: allowed={d.allowed} license={d.license} {d.reason}")
assert d.allowed, f"このモデルは商用利用ポリシーで許可されていません: {d.reason}"

## 3-4. biomni と default_config

**順序が重要**: `apply_biomni_env()` を biomni の import より前に呼ぶ。
`biomni.config.default_config` はモジュール読み込み時に環境変数を読むため、
後から設定しても `biomni/tool/database.py` が Anthropic を呼びに行く。

In [ ]:
applied = apply_biomni_env(settings)   # ← import より前
for k, v in applied.items():
    print(f"{k:24s} = {v}")

In [ ]:
try:
    import biomni
    from biomni.version import __version__ as biomni_version
    print("biomni:", biomni_version)
    BIOMNI_AVAILABLE = True
except ImportError as exc:
    print("biomni 未インストール:", exc)
    print("  pip install biomni langchain-ollama")
    BIOMNI_AVAILABLE = False

In [ ]:
from biomni_hypo.config import assert_biomni_env

if BIOMNI_AVAILABLE:
    assert_biomni_env(settings)     # 落ちたら §4.3 の環境変数設定を見直す
    print("✅ default_config は Ollama / 商用モードを向いています")

    from biomni.config import default_config
    print("   llm            :", default_config.llm)
    print("   source         :", default_config.source)
    print("   commercial_mode:", default_config.commercial_mode)

## 5. データレイク

`A1.__init__` は `expected_data_lake_files` を渡さないと**全ファイルを S3 から取得しようとする**
（数十 GB / docs/design/04 §4.4）。本アプリは許可リストのファイルだけを扱う。

In [ ]:
import pathlib

data_lake = pathlib.Path(settings.data_path) / "biomni_data" / "data_lake"
present = {p.name for p in data_lake.glob("*")} if data_lake.exists() else set()
allowed = policy.allowed_dataset_names()

print(f"data_lake: {data_lake}")
print(f"許可 {len(allowed)} 件中 {len(present & set(allowed))} 件が取得済み\n")
for name in allowed:
    mark = "✓" if name in present else "·"
    d = policy.check_dataset(name)
    flag = " ⚠️要ライセンス確認" if d.review_required else ""
    print(f"  {mark} {name:52s} {d.license}{flag}")

### 取得するには

```bash
python scripts/fetch_datasets.py            # 許可リスト全件
python scripts/fetch_datasets.py --only gwas_catalog.pkl gene_info.parquet
```

データが 0 件でも、ツール（公共 DB への問い合わせ）だけで動くランは実行できる。
まずは `gwas_catalog.pkl` と `gene_info.parquet` だけあれば 02 以降が試せる。

## チェック結果まとめ

In [ ]:
checks = {
    "Ollama 到達": status.reachable,
    f"モデル {settings.model} 取得済み": settings.model in status.models,
    "ポリシー読み込み": policy.version >= 1,
    "モデルがポリシー許可": d.allowed,
    "biomni インストール": BIOMNI_AVAILABLE,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

if not all(checks.values()):
    print("\n❌ の項目を解消してから 01 に進むこと。")
else:
    print("\n➡️  01_ollama_stop_sequence.ipynb へ")